# 01 - Find and price GPUs (read-only)

Explores the public gpu.nz catalogue: GPU models, current prices and VRAM. **No API key needed, nothing is rented or charged.**

Requires: `pip install requests matplotlib` and the `python/` folder of this repository next to this notebook's parent directory.
Prices are market prices in rubles per hour; see https://gpu.nz/pricing.

In [ ]:
import sys
sys.path.insert(0, "../python")  # use the client from this repository

from gpunz_client import GpunzClient, format_money, per_gpu_kop_h, rub_to_usd

client = GpunzClient()  # works without a key for catalogue calls
body = client.list_gpus()
usd_rub = body["fx"]["usd_rub"]
gpus = [g for g in body["data"] if g["from_kop_h"] is not None and g["available_gpus"] > 0]
modern = [g for g in gpus if g["tier"] != "legacy"]  # skip old cards for the recommendations below
print(f"{len(gpus)} GPU models available now ({len(modern)} non-legacy), 1 USD = {usd_rub} RUB")

## Cheapest price per card, by model

In [ ]:
rows = sorted(modern, key=lambda g: g["from_kop_h"])[:15]
print(f"{'slug':<24}{'VRAM':>6}{'free':>6}{'RUB/h':>9}{'USD/h':>9}")
for g in rows:
    usd = rub_to_usd(g["from_kop_h"], usd_rub)
    print(f"{g['slug']:<24}{g['vram_gb']:>6}{g['available_gpus']:>6}{g['from_kop_h'] / 100:>9.2f}{usd:>9.3f}")

## VRAM vs price

One dot per GPU model (cheapest on-demand price per card). Consumer cards sit at the bottom, datacenter cards at the top.

In [ ]:
import matplotlib.pyplot as plt

colors = {"consumer": "#2a9d8f", "workstation": "#e9c46a", "datacenter": "#e76f51", "legacy": "#8d99ae"}
fig, ax = plt.subplots(figsize=(9, 5))
for tier, color in colors.items():
    pts = [g for g in gpus if g["tier"] == tier]
    ax.scatter([g["vram_gb"] for g in pts], [g["from_kop_h"] / 100 for g in pts], label=tier, color=color, alpha=0.85)
for g in gpus:
    if g["slug"] in {"rtx-3090-24gb", "rtx-4090-24gb", "rtx-5090-32gb", "a100-pcie-80gb", "h100-sxm-80gb", "l40s-48gb"}:
        ax.annotate(g["name"], (g["vram_gb"], g["from_kop_h"] / 100), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("VRAM per GPU, GB")
ax.set_ylabel("cheapest price, RUB per hour per card")
ax.set_title("gpu.nz: VRAM vs hourly price (live)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Cheapest non-legacy card per VRAM tier

In [ ]:
for need in (8, 12, 16, 24, 32, 48, 80):
    fit = [g for g in modern if g["vram_gb"] >= need]
    best = min(fit, key=lambda g: g["from_kop_h"], default=None)
    if best:
        print(f">= {need:>2} GB: {best['name']:<22} {format_money(best['from_kop_h'])}/h  ({best['slug']})")

## Real offers for one model

Offer prices are for the **whole server**; divide by the GPU count for a per-card price.

In [ ]:
offers = client.search_offers(gpu="rtx-4090-24gb", num_gpus=1, min_reliability=0.95, order="price", limit=5)
for o in offers:
    print(o["id"], o.get("country"), format_money(o["price"]["on_demand_kop_h"]) + "/h", "reliability", o.get("reliability"))

## What would 3 hours cost?

In [ ]:
from gpunz_client import estimate_cost_minor

if offers:
    kop_h = offers[0]["price"]["on_demand_kop_h"]
    cost = estimate_cost_minor(kop_h, 180)
    print(f"3 h on offer {offers[0]['id']}: {format_money(cost)}  (~{rub_to_usd(cost, usd_rub):.2f} USD)")